In [1]:
from pathlib import Path


import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
Path.cwd()

WindowsPath('c:/Users/VENKATESH/aircraft-rul-predictor/notebooks')

In [3]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
    
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_DIR / 'raw'
CMAPSS_DATA_DIR = RAW_DATA_DIR / 'cmapss' / 'CMAPSSData'

TEST_FILE_PATH = CMAPSS_DATA_DIR / "test_FD001.txt"
RUL_FILE_PATH = CMAPSS_DATA_DIR / "RUL_FD001.txt"

MODEL_PATH = PROJECT_ROOT / "models" / "xgboost_rul_model.joblib"

print("Project root:", PROJECT_ROOT)
print("Test file exists:", TEST_FILE_PATH.exists())
print("RUL file exists:", RUL_FILE_PATH.exists())
print("Model file exists:", MODEL_PATH.exists())

Project root: c:\Users\VENKATESH\aircraft-rul-predictor
Test file exists: True
RUL file exists: True
Model file exists: True


In [4]:
model_artifact = joblib.load(MODEL_PATH)

final_model = model_artifact['model']
feature_columns = model_artifact['feature_columns']
max_rul = model_artifact['max_rul']
model_name = model_artifact['model_name']

print(f'Loaded model is {model_name}')
print(f'Number of features is: {len(feature_columns)}')
print(f'MAX_RUL used during training: {max_rul}')
print("\nFeature columns:")
print(feature_columns)

Loaded model is Tuned XGBoost
Number of features is: 17
MAX_RUL used during training: 125

Feature columns:
['operational_setting_1', 'operational_setting_2', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


In [5]:
index_columns = ['unit_number', 'time_in_cycles']

operational_setting_columns = [
    'operational_setting_1',
    'operational_setting_2',
    'operational_setting_3',
]

sensor_columns = [f'sensor_{num}' for num in range(1,22)]

column_names = index_columns + operational_setting_columns + sensor_columns

print(f'Total columns: {len(column_names)}')
print(column_names)

Total columns: 26
['unit_number', 'time_in_cycles', 'operational_setting_1', 'operational_setting_2', 'operational_setting_3', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21']


In [7]:
test_df = pd.read_csv(
    TEST_FILE_PATH,
    sep = r"\s+",
    header = None,
    names = column_names,
)

test_rul_df = pd.read_csv(
    RUL_FILE_PATH,
    sep = r"\s+",
    header = None,
    names = ['true_rul'],
)


print(f'The shape of the test file is: {test_df.shape}')
print(f'The shape of the test rul file is: {test_rul_df.shape}')


test_df.head()

The shape of the test file is: (13096, 26)
The shape of the test rul file is: (100, 1)


,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,operational_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,0.0023,0.0003,100.0,518.67,643.02,1585.29,1398.21,14.62,...,521.72,2388.03,8125.55,8.4052,0.03,392,2388,100.0,38.86,23.3735
1,1,2,-0.0027,-0.0003,100.0,518.67,641.71,1588.45,1395.42,14.62,...,522.16,2388.06,8139.62,8.3803,0.03,393,2388,100.0,39.02,23.3916
2,1,3,0.0003,0.0001,100.0,518.67,642.46,1586.94,1401.34,14.62,...,521.97,2388.03,8130.10,8.4441,0.03,393,2388,100.0,39.08,23.4166
3,1,4,0.0042,0.0000,100.0,518.67,642.44,1584.12,1406.42,14.62,...,521.38,2388.05,8132.90,8.3917,0.03,391,2388,100.0,39.00,23.3737
4,1,5,0.0014,0.0000,100.0,518.67,642.51,1587.19,1401.92,14.62,...,522.15,2388.03,8129.54,8.4031,0.03,390,2388,100.0,38.99,23.4130


In [15]:
# The RUL file has one row per test engine.
test_rul_df['unit_number'] = range(1, len(test_rul_df) +1)

# For each test engine, select only the last available cycle.
# NASA's true RUL value corresponds to this final observed cycle.
test_last_cycle_df = (
    test_df.sort_values('time_in_cycles')
    .groupby('unit_number')
    .tail(1)
)

# Use the exact same feature columns used in the training
X_test_official = test_last_cycle_df[feature_columns]


# Match each test engin with its true url 
official_test_data = test_last_cycle_df[['unit_number']].merge(
    test_rul_df[['unit_number', 'true_rul']],
    on = 'unit_number',
    how = 'inner',
)

y_test_official = official_test_data['true_rul']

print('Official test input shape', X_test_official.shape)
print('Official test target shape', y_test_official.shape)


Official test input shape (100, 17)
Official test target shape (100,)


In [20]:
official_test_predictions = final_model.predict(X_test_official)

official_test_mae = mean_absolute_error(
    y_test_official,
    official_test_predictions,
)

official_test_rmse = np.sqrt(
    mean_squared_error(
        y_test_official,
        official_test_predictions,
    )
)

official_test_r2 = r2_score(
    y_test_official,
    official_test_predictions,
)

print("Official NASA Test Performance")
print("------------------------------")
print("MAE:", round(official_test_mae, 2))
print("RMSE:", round(official_test_rmse, 2))
print("R2 Score:", round(official_test_r2, 4))


Official NASA Test Performance
------------------------------
MAE: 13.14
RMSE: 17.92
R2 Score: 0.8141


In [21]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

official_results_df = official_test_data.copy()

official_results_df["predicted_rul"] = official_test_predictions

official_results_df["absolute_error"] = (
    official_results_df["true_rul"] - official_results_df["predicted_rul"]
).abs()

official_results_df["prediction_error"] = (
    official_results_df["true_rul"] - official_results_df["predicted_rul"]
)

official_results_df = official_results_df.sort_values(
    by="absolute_error",
    ascending=False,
)

results_file_path = REPORTS_DIR / "official_test_predictions.csv"

official_results_df.to_csv(results_file_path, index=False)

print("Official test results saved at:")
print(results_file_path)

official_results_df.head(10)

Official test results saved at:
c:\Users\VENKATESH\aircraft-rul-predictor\reports\official_test_predictions.csv


,unit_number,true_rul,predicted_rul,absolute_error,prediction_error
54,27,66,115.046684,49.046684,-49.046684
30,79,63,111.809639,48.809639,-48.809639
40,41,18,65.267548,47.267548,-47.267548
65,45,114,68.590889,45.409111,45.409111
98,93,85,41.830051,43.169949,43.169949
21,15,83,119.774582,36.774582,-36.774582
95,12,124,89.083405,34.916595,34.916595
37,37,21,55.691368,34.691368,-34.691368
14,67,77,110.926056,33.926056,-33.926056
23,11,97,64.931351,32.068649,32.068649
